# Algoritmos de búsqueda con oráculo: Deutsch-Jozsa

Este notebook empieza una pareja de algoritmos que comparten una idea central: el oráculo, una función codificada como puerta cuántica que se puede consultar en superposición. Deutsch-Jozsa es el más simple de los dos, y el siguiente notebook, Grover, reutiliza la misma idea.

## El problema

Hay una función `f` de `n` bits a 1 bit, con la garantía de que es **constante**, misma salida para toda entrada, o **balanceada**, la mitad de las entradas dan 0 y la otra mitad dan 1. El objetivo es decidir cuál de las dos es, consultando `f` el menor número de veces posible.

Clásicamente, en el peor caso hace falta consultar más de la mitad de las entradas para estar seguro. Deutsch-Jozsa lo decide con una única consulta, gracias a poder evaluar `f` sobre una superposición de todas las entradas a la vez.

## Qué es un oráculo

Un oráculo es la función `f` codificada como una puerta cuántica reversible: `|x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩`. El registro `x` son los qubits de entrada, e `y` es un qubit auxiliar que acumula el resultado con una XOR.

Preparando el auxiliar en el estado `|−⟩` antes de consultar el oráculo, el resultado de `f(x)` aparece como un cambio de fase sobre el registro de entrada en vez de sumarse al auxiliar. Este truco, llamado retroceso de fase, es lo que permite extraer información de todas las entradas a la vez.

## El algoritmo

1. `n` qubits de entrada en `|0⟩`, un qubit auxiliar en `|1⟩`.
2. H sobre todos los qubits, incluido el auxiliar: esto lo deja en `|−⟩`.
3. Aplicar el oráculo.
4. H de nuevo sobre los qubits de entrada, sin tocar el auxiliar.
5. Medir los qubits de entrada: todo ceros implica `f` constante, cualquier otro resultado implica `f` balanceada.

## Un oráculo constante

La función más simple, `f(x) = 0` para toda entrada, no necesita ninguna puerta: el auxiliar nunca cambia.

In [ ]:
import polypus

qc_constante = polypus.Circuit(3)
qc_constante.x(2)  # auxiliar en |1>
qc_constante.h(0)
qc_constante.h(1)
qc_constante.h(2)
# oraculo: f(x) = 0, ninguna puerta
qc_constante.h(0)
qc_constante.h(1)
qc_constante.measure(0, 0)
qc_constante.measure(1, 1)

result = polypus.run_quantum_circuit(qc_constante, shots=1000, infrastructure="local")
print(result.counts[0])

El resultado es `'00'` el 100% de las veces: la firma de una función constante.

## Un oráculo balanceado

`f(x0, x1) = x0 ⊕ x1` es balanceada: de las cuatro entradas posibles, dos dan 0 y dos dan 1. Se implementa con una CX desde cada qubit de entrada hacia el auxiliar:

In [ ]:
qc_balanceado = polypus.Circuit(3)
qc_balanceado.x(2)
qc_balanceado.h(0)
qc_balanceado.h(1)
qc_balanceado.h(2)
qc_balanceado.cx(0, 2)  # oraculo: f(x0, x1) = x0 xor x1
qc_balanceado.cx(1, 2)
qc_balanceado.h(0)
qc_balanceado.h(1)
qc_balanceado.measure(0, 0)
qc_balanceado.measure(1, 1)

Con el patrón visto en `02b`, el circuito completo:

In [ ]:
from qiskit import qasm2

qc_dibujo = qasm2.loads(
    qc_balanceado.to_qasm2(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS
)
qc_dibujo.draw("mpl")

In [ ]:
result = polypus.run_quantum_circuit(qc_balanceado, shots=1000, infrastructure="local")
print(result.counts[0])

El resultado es `'11'` el 100% de las veces, nunca `'00'`: la firma de una función balanceada. Distinguir entre las dos ha hecho falta con una sola consulta al oráculo, no varias.

## Resumen

Este notebook ha introducido el oráculo como una función cuántica consultable en superposición, y Deutsch-Jozsa como el algoritmo más simple que se aprovecha de eso: distingue una función constante de una balanceada con una sola consulta, donde clásicamente hacen falta varias.

El siguiente notebook, Grover, reutiliza el mismo concepto de oráculo para buscar en una lista sin ordenar.

## Siguiente paso

Continúa con: [`05b_grover.ipynb`](05b_grover.ipynb).